# Korean Chatbot - Stage 1 학습 노트북
KoAlpaca 데이터셋으로 스크래치 Transformer 학습

## 0. 환경 설정

In [ ]:
!pip install datasets tqdm -q

In [ ]:
import os

REPO_URL = "https://github.com/sehoon-kim/korean-chatbot.git"  # 본인 레포로 수정
REPO_DIR = "korean-chatbot"
STAGE_DIR = f"{REPO_DIR}/stage1_from_scratch"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
else:
    !git -C $REPO_DIR pull

import sys
sys.path.insert(0, STAGE_DIR)
print("레포 준비 완료")

## 1. 토크나이저 준비
저장된 tokenizer.json 이 있으면 로드, 없으면 KoAlpaca로 새로 학습

In [ ]:
import sys
sys.path.insert(0, STAGE_DIR)

from src.tokenizer import BPETokenizer
import config

TOKENIZER_SAVE_PATH = "tokenizer.json"

tokenizer = BPETokenizer()

if os.path.exists(TOKENIZER_SAVE_PATH):
    tokenizer.load(TOKENIZER_SAVE_PATH)
    print(f"토크나이저 로드 완료. vocab size = {len(tokenizer.vocab)}")
else:
    from src.tokenizer import train_on_koalpaca
    tokenizer = train_on_koalpaca(
        vocab_size=config.VOCAB_SIZE,
        max_corpus_lines=30_000,   # 전체 코퍼스 대신 3만 줄만 사용 (속도 단축)
        save_path=TOKENIZER_SAVE_PATH
    )
    print(f"토크나이저 학습·저장 완료. vocab size = {len(tokenizer.vocab)}")

## 2. 데이터셋 로드 및 전처리

In [ ]:
from datasets import load_dataset

print("KoAlpaca 데이터셋 로드 중...")
ds = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
print(f"총 샘플 수: {len(ds)}")
print("예시:", ds[0])

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

PAD_ID = tokenizer.vocab["<pad>"]
BOS_ID = tokenizer.vocab["<s>"]
EOS_ID = tokenizer.vocab["</s>"]


class KoAlpacaDataset(Dataset):
    """instruction + output 을 이어붙여 Language Model 방식으로 학습"""

    def __init__(self, hf_dataset, tokenizer, max_seq_len):
        self.samples = []
        self.max_seq_len = max_seq_len

        for row in hf_dataset:
            instruction = row.get("instruction", "").strip()
            inp = row.get("input", "").strip()
            output = row.get("output", "").strip()

            # 형식: <s> 질문\n답변 </s>
            if inp:
                text = f"{instruction}\n{inp}\n{output}"
            else:
                text = f"{instruction}\n{output}"

            ids = [BOS_ID] + tokenizer.encode(text) + [EOS_ID]

            # max_seq_len+1 토큰까지만 (input/label shift 때문에)
            ids = ids[: max_seq_len + 1]
            if len(ids) > 1:   # 너무 짧은 샘플 제외
                self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        # LM: input = ids[:-1], label = ids[1:]
        x = ids[:-1]
        y = ids[1:]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def collate_fn(batch):
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    padded_x = torch.stack([torch.nn.functional.pad(x, (0, max_len - x.size(0)), value=PAD_ID) for x in xs])
    padded_y = torch.stack([torch.nn.functional.pad(y, (0, max_len - y.size(0)), value=PAD_ID) for y in ys])
    return padded_x, padded_y


dataset = KoAlpacaDataset(ds, tokenizer, max_seq_len=config.MAX_SEQ_LEN)
print(f"유효 샘플 수: {len(dataset)}")

dataloader = DataLoader(
    dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)
print(f"배치 수: {len(dataloader)}")

## 3. 모델 초기화

In [ ]:
from src.model import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = Transformer(
    vocab_size=len(tokenizer.vocab),
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    max_seq_len=config.MAX_SEQ_LEN,
    dropout=config.DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"파라미터 수: {total_params:,} ({total_params/1e6:.1f}M)")

## 4. 학습

In [ ]:
import math
from tqdm.notebook import tqdm

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.LR)

# Cosine LR 스케줄러 (선택)
total_steps = len(dataloader) * config.EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

CHECKPOINT_PATH = "checkpoint.pt"
GRAD_CLIP = 1.0

# 이어서 학습하려면 체크포인트 로드
start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    print(f"체크포인트 로드 완료. epoch {start_epoch}부터 재개")


def train_epoch(epoch):
    model.train()
    total_loss = 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{config.EPOCHS}", leave=False)

    for step, (x, y) in enumerate(pbar):
        x, y = x.to(device), y.to(device)

        logits = model(x)                        # (B, T, V)
        loss = criterion(
            logits.view(-1, logits.size(-1)),    # (B*T, V)
            y.view(-1)                           # (B*T,)
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        avg_loss = total_loss / (step + 1)
        pbar.set_postfix({"loss": f"{avg_loss:.4f}", "ppl": f"{math.exp(avg_loss):.1f}"})

    return total_loss / len(dataloader)


print("학습 시작!")
for epoch in range(start_epoch, config.EPOCHS):
    avg_loss = train_epoch(epoch)
    ppl = math.exp(avg_loss)
    print(f"Epoch {epoch+1:02d} | loss: {avg_loss:.4f} | ppl: {ppl:.2f}")

    torch.save(
        {"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict()},
        CHECKPOINT_PATH,
    )

print("학습 완료!")

## 5. 간단한 생성 테스트

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100, temperature=1.0, top_k=50):
    model.eval()
    ids = [BOS_ID] + tokenizer.encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        if x.size(1) >= config.MAX_SEQ_LEN:
            break
        logits = model(x)[:, -1, :]   # 마지막 토큰의 logit
        logits = logits / temperature

        # top-k 샘플링
        if top_k > 0:
            topk_vals, _ = torch.topk(logits, top_k)
            logits[logits < topk_vals[:, -1:]] = float("-inf")

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

        if next_id.item() == EOS_ID:
            break

    generated_ids = x[0].tolist()[len(ids):]  # 프롬프트 제외
    return tokenizer.decode(generated_ids)


prompt = "한국의 수도는 어디인가요?"
print(f"입력: {prompt}")
print(f"생성: {generate(prompt)}")

## 6. Google Drive 저장 (선택)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import shutil
# shutil.copy(CHECKPOINT_PATH, '/content/drive/MyDrive/korean_chatbot_checkpoint.pt')
# shutil.copy(TOKENIZER_SAVE_PATH, '/content/drive/MyDrive/korean_chatbot_tokenizer.json')
# print('Drive 저장 완료')